In [ ]:
import os
import torch
import torch.utils.data
from torch.utils.data import DataLoader
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from PIL import Image
import numpy as np
from pathlib import Path
import cv2

# ========= Dataset Loader =========
def load_yolov8_segmentation(label_path, img_size):
    H, W = img_size
    boxes, labels, masks = [], [], []

    if not os.path.exists(label_path):
        return torch.zeros((0, 4), dtype=torch.float32), \
               torch.zeros((0,), dtype=torch.int64), \
               torch.zeros((0, H, W), dtype=torch.uint8)

    with open(label_path, 'r') as f:
        for line in f:
            data = list(map(float, line.strip().split()))
            if len(data) < 7:
                continue

            cls_id = int(data[0])
            polygon = np.array(data[1:], dtype=np.float32).reshape(-1, 2)
            polygon[:, 0] *= W
            polygon[:, 1] *= H

            if polygon.shape[0] < 3:
                continue

            mask = np.zeros((H, W), dtype=np.uint8)
            cv2.fillPoly(mask, [polygon.astype(np.int32)], 1)
            if mask.sum() == 0:
                continue

            x_min = np.min(polygon[:, 0])
            y_min = np.min(polygon[:, 1])
            x_max = np.max(polygon[:, 0])
            y_max = np.max(polygon[:, 1])
            if x_max <= x_min or y_max <= y_min:
                continue

            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(cls_id + 1)
            masks.append(mask)

    if len(boxes) == 0:
        return torch.zeros((0, 4), dtype=torch.float32), \
               torch.zeros((0,), dtype=torch.int64), \
               torch.zeros((0, H, W), dtype=torch.uint8)

    return torch.tensor(boxes, dtype=torch.float32), \
           torch.tensor(labels, dtype=torch.int64), \
           torch.tensor(np.array(masks), dtype=torch.uint8)


class YOLOv8SegmentationDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, label_dir, transforms=None):
        self.img_dir = Path(img_dir)
        self.label_dir = Path(label_dir)
        self.transforms = transforms
        self.images = list(self.img_dir.glob("*.jpg")) + list(self.img_dir.glob("*.png"))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        while True:
            img_path = self.images[idx]
            img = Image.open(img_path).convert("RGB")
            W, H = img.size
            label_path = self.label_dir / (img_path.stem + ".txt")
            boxes, labels, masks = load_yolov8_segmentation(label_path, (H, W))
            if boxes.shape[0] == 0:
                idx = np.random.randint(0, len(self.images))
                continue

            target = {"boxes": boxes, "labels": labels, "masks": masks}
            if self.transforms:
                img, target = self.transforms(img, target)
            return torchvision.transforms.ToTensor()(img), target


# ========= Model Setup =========
def get_instance_segmentation_model(num_classes):
    model = maskrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)
    return model


# ========= Training Loop with Early Stopping =========
def train_model(model, train_loader, val_loader, optimizer, device, 
                num_epochs=10, early_stop_delta=1e-3, patience=3, max_grad_norm=5.0):
    
    best_loss = float('inf')
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        # ---- Training ----
        model.train()
        train_loss = 0
        for images, targets in train_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Skip batch if empty targets exist
            if any(len(t["boxes"]) == 0 for t in targets):
                continue

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            train_loss += losses.item()

            optimizer.zero_grad()
            losses.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
            optimizer.step()

        avg_train_loss = train_loss / len(train_loader)

        # ---- Validation ----
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, targets in val_loader:
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

                if len(targets) == 0 or all(len(t['boxes']) == 0 for t in targets):
                    continue  # skip empty targets to avoid list output

                # Ensure model returns dict
                model.train()  # temporarily compute losses
                loss_dict = model(images, targets)
                if isinstance(loss_dict, list):
                    # happens if targets are empty, skip
                    continue
                losses = sum(loss for loss in loss_dict.values())
                val_loss += losses.item()
                model.eval()

        avg_val_loss = val_loss / len(val_loader)

        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        # ---- Early Stopping Check ----
        if best_loss - avg_val_loss > early_stop_delta:
            best_loss = avg_val_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break


# ========= Main =========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_image_dir = '/home/viny/Desktop/Sudoku-Solver/Sudoku.v1i.yolov8/train/images'
train_label_dir = '/home/viny/Desktop/Sudoku-Solver/Sudoku.v1i.yolov8/train/labels'
val_image_dir   = '/home/viny/Desktop/Sudoku-Solver/Sudoku.v1i.yolov8/test/images'
val_label_dir   = '/home/viny/Desktop/Sudoku-Solver/Sudoku.v1i.yolov8/test/labels'

train_dataset = YOLOv8SegmentationDataset(train_image_dir, train_label_dir)
val_dataset   = YOLOv8SegmentationDataset(val_image_dir, val_label_dir)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader   = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

num_classes = 2  # background + sudoku
model = get_instance_segmentation_model(num_classes)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=2e-4, momentum=0.9, weight_decay=0.0005)

train_model(model, train_loader, val_loader, optimizer, device, num_epochs=10, early_stop_delta=1e-3, patience=3)

torch.save(model.state_dict(), "maskrcnn_sudoku.pth")


Epoch [1/10] Train Loss: 0.6735 | Val Loss: 0.3000
Epoch [2/10] Train Loss: 0.1765 | Val Loss: 0.2598
Epoch [3/10] Train Loss: 0.1339 | Val Loss: 0.1781
Epoch [4/10] Train Loss: 0.1176 | Val Loss: 0.1650
Epoch [5/10] Train Loss: 0.1079 | Val Loss: 0.1582
Epoch [6/10] Train Loss: 0.1017 | Val Loss: 0.1495
Epoch [7/10] Train Loss: 0.0975 | Val Loss: 0.1431
Epoch [8/10] Train Loss: 0.0923 | Val Loss: 0.1384
Epoch [9/10] Train Loss: 0.0897 | Val Loss: 0.1324
Epoch [10/10] Train Loss: 0.0866 | Val Loss: 0.1302


In [2]:
import cv2
import torch
import numpy as np
from torchvision.transforms import ToTensor

def segment_sudoku(image_path, model, device, threshold=0.5):
    """
    Segments Sudoku grids in an image using a trained Mask R-CNN model.
    
    Args:
        image_path (str): Path to input image.
        model (torch.nn.Module): Trained Mask R-CNN model.
        device (torch.device): torch.device('cuda') or torch.device('cpu')
        threshold (float): Minimum score for considering a detection.
    
    Returns:
        output_img (np.ndarray): Original image with masks drawn.
    """
    # Read image with OpenCV
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape

    # Convert to tensor and add batch dimension
    img_tensor = ToTensor()(img_rgb).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model([img_tensor])
    
    # Get masks, scores, and labels
    masks = outputs[0]['masks']  # (N, 1, H, W)
    scores = outputs[0]['scores']
    
    # Filter by threshold
    masks = masks[scores > threshold]
    
    if len(masks) == 0:
        return img  # nothing detected
    
    # Combine masks into a single mask
    combined_mask = torch.zeros((h, w), dtype=torch.uint8).to(device)
    for mask in masks:
        mask_binary = (mask[0] > 0.5).byte()  # threshold mask
        combined_mask = torch.max(combined_mask, mask_binary)
    
    combined_mask = combined_mask.cpu().numpy() * 255
    
    # Create colored mask overlay
    colored_mask = np.zeros_like(img)
    colored_mask[:, :, 1] = combined_mask  # green mask
    alpha = 0.5
    output_img = cv2.addWeighted(img, 1, colored_mask, alpha, 0)
    
    return output_img


In [4]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Load trained model
num_classes = 2
model = get_instance_segmentation_model(num_classes)
model.load_state_dict(torch.load("maskrcnn_sudoku.pth", map_location=device))
model.to(device)

# Segment image
result = segment_sudoku('/home/viny/Desktop/Sudoku-Solver/Sudoku.v1i.yolov8/test/images/12d8cbcc-image168_jpg.rf.f46b55161d27f88b24128bad4abbf212.jpg', model, device)

# Show result
cv2.imshow("Sudoku Segmentation", result)
cv2.waitKey(0)
cv2.destroyAllWindows()

/tmp/ipykernel_34305/3860581000.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("maskrcnn_sudoku.pth", map_location=device))


KeyboardInterrupt: 